# GRF microstructure and SSA

This notebook generates two-dimensional binary snow microstructures with the `grfsaw` package and estimates their specific surface area (SSA) from spatial autocorrelation. The examples compare structures with different target SSA values while keeping the solid fraction fixed.

## Workflow
1. Define an SSA calculation from the binary structure's autocorrelation and convert the result to mass-specific units.
2. Generate the example microstructures and calculate their SSA.
3. Plot the structures and inspect the estimates.

The notebook expects `grfsaw` to be installed and writes its example figure to the path specified in the plotting cell. Adjust that path for your environment.

In [ ]:
# %matplotlib inline
# %matplotlib notebook
import numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.append('/home/jum002/store7/repo/GRFsaw')

from grfsaw.microstructure import Microstructure

In [ ]:
from grfsaw import autocorrelation_fft

def calculate_ssa(binary, num_mesh_points, unit_size):
    phi_observed = np.sum(binary) / num_mesh_points

    autocorr_r, dist_autocorr_r = autocorrelation_fft(binary)
    autocorr_r *= phi_observed

    ### surface area per unit total (solid+void) volume
    SSA = -4 * ( autocorr_r[1] - autocorr_r[0] )/unit_size # Note that dist_autocorr_r[1] - dist_autocorr_r[0] = 1 - 0 = 1

    ### convert from "surface area per unit total volume" to "surface area per unit solid volume"
    SSA /= phi_observed 

    return SSA/ 917

In [ ]:
shape = (200,200)
list_m = [(8, 3), (16, 8), (24, 12)]
list_phi = [0.3, 0.3, 0.3]

micros = []
for m, phi in zip(list_m, list_phi):
    micro = Microstructure(shape, phi, single_cut = True, distr = 'g', m_mean = m[0], m_std = m[1], folder = "grf_output/", procs = 64)
    micro.create()
    micros.append(micro)
    print(calculate_ssa(micro.binary_array, micro.num_mesh_points, 0.00005))

In [ ]:
fig, ax = plt.subplots(1,3, figsize = (8,4),sharex=True, sharey=True)

ax[0].imshow(micros[0].binary_array.T, cmap='Greys', interpolation='nearest', origin='lower')
ax[0].contour(micros[0].binary_array.T, levels=[0.5], colors='C1', linewidths=2)
ax[1].imshow(micros[1].binary_array.T, cmap='Greys', interpolation='nearest', origin='lower')
ax[1].contour(micros[1].binary_array.T, levels=[0.5], colors='C1', linewidths=1)
ax[2].imshow(micros[2].binary_array.T, cmap='Greys', interpolation='nearest', origin='lower')
ax[2].contour(micros[2].binary_array.T, levels=[0.5], colors='C1', linewidths=0.75)

ax[0].set_title('a) $SSA_M$ = 8 m$^2$ kg$^{-1}$')
ax[1].set_title('b) $SSA_M$ = 16 m$^2$ kg$^{-1}$')
ax[2].set_title('c) $SSA_M$ = 24 m$^2$ kg$^{-1}$')


ax[0].set_xticks(np.arange(0, 200, 20))
ax[0].set_yticks(np.arange(0, 200, 20))
ax[0].set_axisbelow(True)

ax[0].grid(alpha=0.5, color = 'k')
ax[0].tick_params(left=False, bottom=False)
ax[1].grid(alpha=0.5, color = 'k')
ax[1].tick_params(left=False, bottom=False)
ax[2].grid(alpha=0.5, color = 'k')
ax[2].tick_params(left=False, bottom=False)

ax[0].set_xticklabels([])
ax[0].set_yticklabels([])
ax[0].set_yticklabels([])


fig.tight_layout()
# fig.savefig('/home/jum002/public_html/ssa/microstructure_example_GRF.png', dpi = 300, bbox_inches='tight', pad_inches=0.1)

In [ ]:
calculate_ssa(micros[0].binary_array, micros[0].num_mesh_points, 0.00005)

In [ ]:
calculate_ssa(micros[1].binary_array, micros[1].num_mesh_points, 0.00005)

In [ ]:
calculate_ssa(micros[2].binary_array, micros[2].num_mesh_points, 0.00005)